# 問題
問題77の学習において、単語埋め込みのパラメータも同時に更新するファインチューニングを導入せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from gensim.models import KeyedVectors
from torch.utils.data import TensorDataset, DataLoader
# ----------------------------
# Device（MPS OOM 回避用に、まずは CPU を推奨）
# ----------------------------
# 強制的にCPUにしたい場合は以下の1行を使う:
# device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("device:", device)

# ----------------------------
# 0) まず SST-2 から「出現語セット」を作る
# ----------------------------
def collect_task_vocab(paths):
    vocab_set = set()
    for path in paths:
        with open(path, "r", encoding="utf-8") as f:
            for raw in f:
                row = raw.strip().split("\t")
                if len(row) < 2 or row[1] not in ("0", "1"):
                    continue
                text = row[0]
                for w in text.strip().split():
                    vocab_set.add(w)
    return vocab_set

PATH_W2V   = "./GoogleNews-vectors-negative300.bin"
PATH_TRAIN = "./SST-2/train.tsv"
PATH_DEV   = "./SST-2/dev.tsv"

task_vocab = collect_task_vocab([PATH_TRAIN, PATH_DEV])
print(f"Unique tokens in SST-2: {len(task_vocab):,}")

# ----------------------------
# 1) word2vec を読み込み（CPU）、SST-2に出る語だけ抽出
#    ※ KeyedVectors 全体はCPUメモリに置かれます。
# ----------------------------
from gensim.models import KeyedVectors
print("Loading word2vec (CPU RAM)...")
w2v = KeyedVectors.load_word2vec_format(PATH_W2V, binary=True)
d_emb = w2v.vector_size

# 2) SST-2 語彙 ∩ word2vec 語彙 を ID 付け
word2id = {"<PAD>": 0}
id2word = {0: "<PAD>"}
kept_words = []
for w in task_vocab:
    if w in w2v:           # OOV は捨てる
        word2id[w] = len(word2id)
        id2word[word2id[w]] = w
        kept_words.append(w)

V = len(word2id)   # PAD 含む
print(f"Kept vocab for SST-2: {V:,} (incl. PAD)")

# 3) 埋め込み行列 E を小さく作成（V × d_emb）
import numpy as np
E = np.zeros((V, d_emb), dtype=np.float32)   # E[0] は PAD=0 のまま
for w in kept_words:
    i = word2id[w]
    E[i] = w2v[w]

# 以降、w2v は不要なら解放してOK
del w2v

# ----------------------------
# 4) SST-2 を ID 化＆パディング（小さい語彙で）
# ----------------------------
def sst_build_batch(path: str, word2id: dict, pad_id: int = 0):
    examples = []
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            row = raw.strip().split("\t")
            if len(row) < 2:
                continue
            label = row[1]
            if label not in ("0", "1"):
                continue
            text = row[0]
            tokens = text.strip().split()
            ids = [word2id[w] for w in tokens if w in word2id]
            if not ids:     # 全部OOVならスキップ
                continue
            examples.append({"ids": ids, "label": float(label)})

    if not examples:
        raise RuntimeError(f"No valid examples parsed from {path}")

    # 長い順にソートして最大長に合わせてパディング
    examples.sort(key=lambda x: len(x["ids"]), reverse=True)
    max_len = len(examples[0]["ids"])

    padded_ids = []
    labels = []
    for ex in examples:
        ids = ex["ids"]
        pad_len = max_len - len(ids)
        if pad_len > 0:
            ids = ids + [pad_id] * pad_len
        padded_ids.append(ids)
        labels.append([ex["label"]])

    input_tensor = torch.tensor(padded_ids, dtype=torch.long)
    label_tensor = torch.tensor(labels, dtype=torch.float32)
    return {"input_ids": input_tensor, "label": label_tensor}

train_batch = sst_build_batch(PATH_TRAIN, word2id, pad_id=0)
dev_batch   = sst_build_batch(PATH_DEV,   word2id, pad_id=0)

# ----------------------------
# 5) 以降はそのまま：
#    - AvgEmbClassifier(E, padding_idx=0, freeze=False)
#    - DataLoader, 学習ループ, 評価
# ----------------------------

# 例：モデル定義（埋め込みをこの小さいEで）
import torch.nn as nn

class AvgEmbClassifier(nn.Module):
    def __init__(self, E_numpy: np.ndarray, pad_id: int = 0, dropout: float = 0.1):
        super().__init__()
        E_torch = torch.tensor(E_numpy, dtype=torch.float32)
        self.emb = nn.Embedding.from_pretrained(E_torch, freeze=False, padding_idx=pad_id)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.emb.weight.size(1), 1)

    def forward(self, ids: torch.Tensor):
        mask = (ids != self.emb.padding_idx)
        x = self.emb(ids)                         # (N, L, d)
        denom = mask.sum(dim=1, keepdim=True).clamp_min(1)
        x = (x * mask.unsqueeze(-1)).sum(dim=1) / denom
        x = self.dropout(x)
        return self.fc(x).squeeze(1)

model = AvgEmbClassifier(E, pad_id=0, dropout=0.1).to(device)

# 軽量オプティマイザ（Adam → SGD）＋ バッチ小さめ
from torch.utils.data import TensorDataset, DataLoader
train_ds = TensorDataset(train_batch["input_ids"], train_batch["label"].squeeze(1))
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, pin_memory=False)

opt  = torch.optim.SGD([
    {"params": model.emb.parameters(), "lr": 5e-4},    # 埋め込みは小さめ
    {"params": model.fc.parameters(),  "lr": 1e-2, "weight_decay": 1e-4},
], momentum=0.9)
crit = nn.BCEWithLogitsLoss()

# 学習
model.train()
for epoch in range(1, 6):  # とりあえず5エポック
    total = 0.0
    for ids, y in train_loader:
        ids = ids.to(device)
        y   = y.to(device).float()
        opt.zero_grad()
        loss = crit(model(ids), y)
        loss.backward()
        opt.step()
        total += loss.item() * ids.size(0)
    print(f"[Epoch {epoch:02d}] loss={total/len(train_ds):.4f}")

# 評価
model.eval()
with torch.no_grad():
    ids_dev = dev_batch["input_ids"].to(device)
    y_dev   = dev_batch["label"].squeeze(1).long().to(device)
    prob = torch.sigmoid(model(ids_dev))
    pred = (prob >= 0.5).long()
    acc  = (pred == y_dev).float().mean().item()
print(f"dev acc: {acc:.3f}")

device: mps
Unique tokens in SST-2: 15,756
Loading word2vec (CPU RAM)...
Kept vocab for SST-2: 13,069 (incl. PAD)
[Epoch 01] loss=0.4662
[Epoch 02] loss=0.3966
[Epoch 03] loss=0.3809
[Epoch 04] loss=0.3693
[Epoch 05] loss=0.3627
dev acc: 0.805
